# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll use the `record_sets` attribute and list their `@id` and basic structure. Then, for each record set, we'll display their fields and the corresponding field `@id`s.

In [ ]:
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets detected automatically. Attempting to scan available schema objects for record sets...")
    # In some datasets, record_sets may be empty but fields and sources may be available via .metadata
    schema_json = dataset.metadata.to_json()
    record_set_list = schema_json.get('recordSet', [])
    if record_set_list:
        # It's possible, per Croissant, these are simply lists of @id strings
        print(f"Record set listing by @id: {[x['@id'] if isinstance(x, dict) and '@id' in x else x for x in record_set_list]}")
    else:
        print("No record sets found in the schema. Please check dataset schema definition.")
else:
    print(f"Record sets found: {[rs['@id'] for rs in record_sets]}")

# We will list some example record set and field IDs:
print("\nDetailed record set and field overview using dataset.record_sets:")
for record_set in record_sets:
    rec_id = record_set['@id']
    print(f"- Record Set @id: {rec_id}")
    if 'field' in record_set and isinstance(record_set['field'], list):
        print("    Fields in this record set:")
        for field in record_set['field']:
            if isinstance(field, dict) and '@id' in field:
                print(f"        Field @id: {field['@id']}   Name: {field.get('name','')}  DataType: {field.get('dataType','')}")
    else:
        print("    (Fields not found in record set definition)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this example, we select the first available record set, assuming the main record set includes protein abundance and related fields.

In [ ]:
# If no record_sets are discovered, the schema may require knowledge of the main table's @id.
if not record_sets:
    # For demonstration, insert an example main record set @id as found (if you know one)
    main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # This id corresponds to a distribution, possibly a file
    print(f"Using distribution id as record set id: {main_record_set_id}")
else:
    main_record_set_id = record_sets[0]['@id']

record_set_ids = [main_record_set_id]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[record_set_id])} rows from record set {record_set_id}")
            print("Columns:", dataframes[record_set_id].columns.tolist())
            display(dataframes[record_set_id].head())
        else:
            print(f"No records loaded for record set {record_set_id}")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field for analysis, for example 'cr:coverage_percentage' if present
df = dataframes.get(main_record_set_id)
if df is not None and not df.empty:
    # Find numeric columns for demonstration
    numeric_candidates = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidates.append(col)
    print(f"Available numeric fields: {numeric_candidates}")
    # Pick the first by default, or specify if you know the id
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Chose {numeric_field} for analysis.")
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 1
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by another field, for example, accession or description if available
        group_field = None
        for col in df.columns:
            if 'accession' in col or 'description' in col or 'group' in col:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No obvious group field found.")
    else:
        print("No numeric columns found in this DataFrame.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_candidates:
    # Plot distribution of selected numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If more than one numeric field, show a pairplot
    if len(numeric_candidates) > 1:
        pair_fields = numeric_candidates[:2]
        plt.figure(figsize=(6, 6))
        sns.scatterplot(data=df, x=pair_fields[0], y=pair_fields[1])
        plt.title(f"Scatter: {pair_fields[0]} vs {pair_fields[1]}")
        plt.show()
else:
    print("No suitable numeric columns to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load and explore a protein abundance dataset using the FAIR-compliant `mlcroissant` library, referencing all entities using their `@id` as per the Croissant schema.

- We reviewed the available record sets and their field IDs.
- We loaded the data into pandas DataFrames, filtered and normalized numeric fields, and visualized distributions.

This workflow provides a reproducible template for further statistical or machine learning analyses on FAIR datasets defined by Croissant schemas.